# 第9回：前処理をPipelineにまとめる

**今日の問い：数値列とカテゴリ列を、安全に同じモデルへ入れるにはどうするか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 列型ごとの前処理を説明する
- 前処理とモデルをPipelineとして一体化する
- 未知カテゴリと欠損を安全に扱う

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- 欠損補完：欠けた値を規則に基づいて埋める処理
- 標準化：尺度を平均0・標準偏差1付近へ揃える処理
- One-Hot：カテゴリを0/1列へ変換する処理
- Pipeline：順序付き処理を1つの推定器として扱う仕組み

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
categorical = ["solvent", "catalyst", "scaffold_group"]
X = df[numeric + categorical]
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## TRY：列ごとの前処理を組み立てる


In [ ]:
numeric_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="median")),
    ("標準化", StandardScaler()),
])
categorical_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("数値列", numeric_process, numeric),
    ("カテゴリ列", categorical_process, categorical),
])
model = Pipeline([
    ("前処理", preprocess),
    ("予測", LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
print(classification_report(y_valid, model.predict(X_valid), target_names=["非活性", "活性"]))


## 未知カテゴリでも予測できるか


In [ ]:
unknown = X_valid.iloc[[0]].copy()
unknown["solvent"] = "New-Solvent"
print("未知カテゴリを含む予測:", model.predict(unknown)[0])


## CHANGE

数値の欠損補完を`median`から`mean`へ変えます。変更はPipelineの1行だけにし、同じ検証データで比べます。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- fit時に学ぶ値とtransformだけの処理を区別する
- 変換後は元より列数が増えることがある
- Pipeline全体を交差検証へ渡す


In [ ]:
transformed_names = model.named_steps["前処理"].get_feature_names_out()
transformed = model.named_steps["前処理"].transform(X_train.head(3))
print("元の列数:", X_train.shape[1], "変換後の列数:", transformed.shape[1])
display(pd.DataFrame(transformed, columns=transformed_names, index=X_train.head(3).index).iloc[:, :12].round(2))


## よくある誤り

- 全データ平均で欠損補完する
- カテゴリを意味のない大小関係へ変換する
- 本番の未知カテゴリでエラーになる

## SELF-STUDY（任意・30〜60分）

- 変換後の特徴量名と列数を確認する
- Pipelineあり・なしの手順を図にしてリーク箇所を示す

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 数値列とカテゴリ列で何を変えるか
2. Pipelineがリークを防ぎやすい理由は何か
3. handle_unknownが必要なのはなぜか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
